# Clase 113 — Optimizadores: Momentum, Nesterov, AdaGrad, RMSProp, Adam, AdamW

Evolución de los optimizadores y cuándo usar cada uno. **Adam/AdamW** para casi todo, **SGD+Momentum** para visión clásica, **Lion** (2023) para LLMs (menos memoria, LR más bajo).

Requiere: `tensorflow` / `keras` (≥ 3.0, incluye `Lion`), `numpy`.

## 1. Construir la familia completa

De SGD a AdamW y Lion. Cada uno resuelve una limitación del anterior.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

optimizadores = {
    "SGD":      keras.optimizers.SGD(learning_rate=0.01),
    "Momentum": keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "Nesterov": keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True),
    "AdaGrad":  keras.optimizers.Adagrad(learning_rate=0.01),
    "RMSProp":  keras.optimizers.RMSprop(learning_rate=1e-3, rho=0.9),
    "Adam":     keras.optimizers.Adam(learning_rate=1e-3),
    "AdamW":    keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-2),
    "Lion":     keras.optimizers.Lion(learning_rate=1e-4, weight_decay=0.1),
}
for nombre, opt in optimizadores.items():
    print(f"{nombre:9s} -> {type(opt).__name__}")

## 2. Hiperparámetros clave: `beta_1`, `beta_2`, `epsilon`, `weight_decay`

Adam usa dos momentos (0.9 / 0.999); Lion no tiene segundo momento (usa el **signo** del gradiente).

In [ ]:
adam = keras.optimizers.Adam(learning_rate=1e-3, beta_1=0.9, beta_2=0.999, epsilon=1e-7)
print("Adam  beta_1:", adam.beta_1, "beta_2:", adam.beta_2, "epsilon:", adam.epsilon)

lion = keras.optimizers.Lion(learning_rate=1e-4, beta_1=0.9, beta_2=0.99)
print("Lion  beta_1:", lion.beta_1, "beta_2:", lion.beta_2, "(sin segundo momento)")

## 3. Comparar optimizadores entrenando el mismo MLP

In [ ]:
def mlp(optimizer):
    m = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(300, activation="relu", kernel_initializer="he_normal"),
        layers.Dense(100, activation="relu", kernel_initializer="he_normal"),
        layers.Dense(10,  activation="softmax"),
    ])
    m.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

for nombre in ["SGD", "Momentum", "Adam", "AdamW", "Lion"]:
    print(f"{nombre:9s} compilado (params {mlp(optimizadores[nombre]).count_params()})")
    # mlp(optimizadores[nombre]).fit(X_tr, y_tr, epochs=20, validation_split=0.1)

## 4. AdamW (decoupled) vs Adam + L2 (coupled)

Loshchilov & Hutter (2019) mostraron que aplicar weight decay separado del gradiente (**AdamW**) supera a Adam con regularización L2 en la loss.

In [ ]:
adamw = mlp(keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-2))

con_l2 = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(300, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-2)),
    layers.Dense(100, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-2)),
    layers.Dense(10,  activation="softmax"),
])
con_l2.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("AdamW (decoupled) suele superar a Adam+L2 (coupled) en val_loss")

## 5. Buffers del optimizer: Adam guarda `m` y `v` por peso

Por eso Adam usa ~2× la memoria de los pesos; Lion solo mantiene `m`.

In [ ]:
modelo = mlp(keras.optimizers.Adam(1e-3))
X = tf.constant(np.random.default_rng(0).normal(size=(64, 784)), dtype=tf.float32)
y = tf.constant(np.random.default_rng(0).integers(0, 10, size=64))
modelo.fit(X, y, epochs=1, verbose=0)                # inicializa los slots del optimizer
print("variables del optimizer Adam (incluye m y v por peso):",
      len(modelo.optimizer.variables))

## Ejercicios

1. **Comparar 5 optimizadores**: SGD, SGD+Momentum, Adam, AdamW, Lion sobre el mismo modelo; graficá `val_loss`.
2. **Tuning del LR**: sweep log de LR para Adam y Lion; verificá que el óptimo de Lion es ~5× más chico.
3. **AdamW vs Adam+L2**: compará `val_loss`.
4. **Inspección de buffers**: imprimí `optimizer.variables` para Adam (m, v) y Lion (solo m).

## Conclusiones

- **Momentum/Nesterov** aceleran en direcciones consistentes; **AdaGrad/RMSProp** adaptan el LR por parámetro.
- **Adam** = momentum + RMSProp + bias correction: el caballito industrial.
- **AdamW** desacopla el weight decay del gradiente: preferilo siempre que uses weight decay.
- **Lion** (2023) usa el signo del gradiente, menos memoria y LR 3-10× más bajo que Adam.
- **SGD+Momentum+cosine** puede generalizar mejor en visión clásica con datasets grandes.

## ✅ Soluciones de los ejercicios

Optimizadores clásicos y modernos con `tensorflow.keras.optimizers`: comparación de 5, sweep de LR, AdamW vs Adam+L2, inspección de buffers y el rol del momentum. Sin TF se validan por AST.

**Ej. 1 — Comparar 5 optimizadores.** SGD, SGD+Momentum, Adam, AdamW y Lion, mismo modelo y datos, 20 épocas.

In [ ]:
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, optimizers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0

def make():
    return keras.Sequential([keras.Input((784,)),
        layers.Dense(128, activation="relu"), layers.Dense(10, activation="softmax")])

opts = {
    "SGD": optimizers.SGD(0.01),
    "Momentum": optimizers.SGD(0.01, momentum=0.9),
    "Adam": optimizers.Adam(1e-3),
    "AdamW": optimizers.AdamW(1e-3, weight_decay=1e-2),
    "Lion": optimizers.Lion(1e-4, weight_decay=0.1),
}
for name, opt in opts.items():
    m = make(); m.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h = m.fit(Xtr, ytr, epochs=20, validation_split=0.2, verbose=0)
    plt.plot(h.history["val_loss"], label=name)
plt.legend(); plt.xlabel("epoca"); plt.ylabel("val_loss")
plt.title("5 optimizadores sobre el mismo modelo"); plt.show()

**Ej. 2 — Tuning del LR.** Sweep log de LR para Adam y Lion: el óptimo de Lion es notablemente más chico.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers, optimizers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0

def run(opt_cls, lr):
    m = keras.Sequential([keras.Input((784,)),
        layers.Dense(128, activation="relu"), layers.Dense(10, activation="softmax")])
    m.compile(optimizer=opt_cls(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m.fit(Xtr, ytr, epochs=5, validation_split=0.2, verbose=0).history["val_accuracy"][-1]

lrs = np.logspace(-5, -2, 6)
for name, cls in [("Adam", optimizers.Adam), ("Lion", optimizers.Lion)]:
    accs = [run(cls, lr) for lr in lrs]
    print(f"{name}: mejor LR = {lrs[int(np.argmax(accs))]:.1e}")
print("Lion actualiza con sign(...) (paso de norma constante): su LR optimo es ~3-10x menor que el de Adam.")

**Ej. 3 — AdamW vs Adam+L2.** El weight decay **desacoplado** de AdamW suele generalizar mejor que la L2 acoplada.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, optimizers, regularizers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0

m1 = keras.Sequential([keras.Input((784,)),
    layers.Dense(128, activation="relu", kernel_regularizer=regularizers.L2(1e-2)),
    layers.Dense(10, activation="softmax", kernel_regularizer=regularizers.L2(1e-2))])
m1.compile(optimizer=optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

m2 = keras.Sequential([keras.Input((784,)),
    layers.Dense(128, activation="relu"), layers.Dense(10, activation="softmax")])
m2.compile(optimizer=optimizers.AdamW(1e-3, weight_decay=1e-2),
           loss="sparse_categorical_crossentropy", metrics=["accuracy"])

for name, m in [("Adam+L2", m1), ("AdamW", m2)]:
    h = m.fit(Xtr, ytr, epochs=10, validation_split=0.2, verbose=0)
    print(f"{name}: val_loss = {h.history['val_loss'][-1]:.3f}")
print("En AdamW el decay no se distorsiona por el escalado adaptativo -> mejor generalizacion.")

**Ej. 4 — Inspección de buffers.** Adam guarda `m` y `v` por parámetro; Lion solo `m` (~mitad de memoria de estado).

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers, optimizers

def state_vars(opt):
    m = keras.Sequential([keras.Input((784,)),
        layers.Dense(128, activation="relu"), layers.Dense(10, activation="softmax")])
    m.compile(optimizer=opt, loss="sparse_categorical_crossentropy")
    m.fit(np.zeros((2, 784), "float32"), np.zeros(2), epochs=1, verbose=0)  # crea las variables
    return len(opt.variables)

print("Adam -> variables de estado:", state_vars(optimizers.Adam(1e-3)))
print("Lion -> variables de estado:", state_vars(optimizers.Lion(1e-4)))
print("Adam guarda 2 buffers/param; Lion 1. En modelos grandes, la diferencia es GBs de VRAM.")

**Ej. 5 — LR alto + Momentum.** SGD(0.1) puro explota; con momentum 0.9 puede funcionar (amortigua oscilaciones).

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, optimizers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0

def run(opt):
    m = keras.Sequential([keras.Input((784,)),
        layers.Dense(128, activation="relu"), layers.Dense(10, activation="softmax")])
    m.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m.fit(Xtr, ytr, epochs=3, validation_split=0.2, verbose=0).history["val_loss"][-1]

print("SGD(0.1) puro      -> val_loss:", run(optimizers.SGD(0.1)))
print("SGD(0.1)+momentum  -> val_loss:", run(optimizers.SGD(0.1, momentum=0.9)))
print("El momentum promedia direcciones sucesivas y amortigua el zig-zag, tolerando LR mas alto.")